# Export the standard radial DOP853 reference

This reproducible export uses the existing checksummed 2× reference. It copies saved samples through normalized time 35 without integrating. Reruns verify an existing export without overwriting it. Run with the project `.venv` kernel.

In [1]:
from pathlib import Path
from copy import deepcopy
import numpy as np
from diagnostics.paths import find_project_root
from diagnostics import load_reference_trajectory, write_reference_trajectory

ROOT = find_project_root(Path.cwd())
SOURCE_DIRECTORY = ROOT / "outputs/developements/accuracy/h5_three_radial_2x_precision/v1"
OUTPUT_DIRECTORY = ROOT / "data/trajectory/h5_three_radial_dop853_t35"
END_TIME = 35.0  # Normalized time, not an oscillation-cycle count.
source = load_reference_trajectory(SOURCE_DIRECTORY)
selected = source.times <= END_TIME
times = source.times[selected]
states = source.states[:, selected]
audit_states = source.audit_states[:, selected]
distances = source.audit_distances[:, selected]
assert times[0] == 0.0 and times[-1] == END_TIME and times.size == 3501
metadata = deepcopy(dict(source.metadata))
metadata.pop("trajectory_sha256", None)
metadata.pop("created_at", None)
metadata["source_artifact"] = {
    "directory": str(SOURCE_DIRECTORY.relative_to(ROOT)),
    "trajectory_sha256": source.metadata["trajectory_sha256"],
    "created_at": source.metadata["created_at"],
    "config": deepcopy(source.metadata["config"]),
    "solver_summary_scope": "Inherited solve summaries describe the full source integration to time 200.",
}
metadata["config"]["t_span"] = [float(times[0]), float(times[-1])]
metadata["reference_name"] = "h5_three_radial_dop853_t35"
metadata["source_notebook"] = "notebooks/developements/accuracy/h5_three_radial_reference_2x_precision/export_standard_reference_t35.ipynb"
metadata["standard_reference"] = True
metadata["arithmetic"] = "IEEE 754 float64; arithmetic precision is distinct from integration error."
metadata["extraction"] = "Exact saved-sample prefix; no integration, interpolation or resampling."
metadata["audit"] = {
    "global_rms_distance": float(np.sqrt(np.mean(distances**2))),
    "maximum_distance": float(distances.max()),
    "final_rms_distance": float(np.sqrt(np.mean(distances[:, -1]**2))),
    "final_maximum_distance": float(distances[:, -1].max()),
    "maximum_state_component_difference": float(np.max(np.abs(states-audit_states))),
}
metadata["audit_maximum_distance_per_particle"] = distances.max(axis=1).tolist()
explanation = f"""# Standard DOP853 reference through normalized time 35

Exact prefix copied from outputs/developements/accuracy/h5_three_radial_2x_precision/v1.
DOP853 states are the standard trajectory. Radau states and periodic distances
are retained solely for the independent numerical audit. No solver was rerun.

The interval is normalized time [0, 35], not 35 oscillation cycles, with 3501
samples spaced by 0.01. State layout: [x1, x2, x3, y1, y2, y3], shape (6, 3501).
Positions use the source characteristic-length normalization (0.06 m).
The source HDF5 potential is PHI_2.h5, B=1.5, indx=(0,1), cubic interpolation;
rho=0.3 and initial radial fractions=(0.1,0.2,0.3), angle=0.

Arithmetic: float64. DOP853 rtol=5e-13, atol=5e-15, maximum step=0.0025.
Radau audit rtol=5e-14, atol=5e-16, maximum step=0.00125.
Inherited solver work and runtime summaries describe the full source run to 200.

Maximum periodic DOP853–Radau discrepancy over this prefix: {distances.max():.12e}.
Per-particle maxima: {distances.max(axis=1).tolist()}.
These are empirical discrepancies, not rigorous trajectory-error bounds.
No independent analytic solution is available for this interpolated measured field.

Load with diagnostics.load_reference_trajectory(directory); use .times and
.states for the DOP853 reference. The loader verifies the array checksum.
Keep trajectory.npz, metadata.json and README.md together. Use only with matching
physical settings, initial states and field fingerprint, within the saved interval.
Reproduce the export with export_standard_reference_t35.ipynb in the 2x notebook folder.
"""
if OUTPUT_DIRECTORY.exists():
    reference = load_reference_trajectory(OUTPUT_DIRECTORY)
else:
    reference = write_reference_trajectory(
        output_directory=OUTPUT_DIRECTORY, times=times, states=states,
        initial_state=source.initial_state, audit_states=audit_states,
        audit_distances=distances, metadata=metadata, explanation=explanation)
for actual, expected in ((reference.times,times),(reference.states,states),
                         (reference.audit_states,audit_states),(reference.audit_distances,distances)):
    np.testing.assert_array_equal(actual, expected)
assert reference.metadata["source_artifact"]["trajectory_sha256"] == source.metadata["trajectory_sha256"]
print("Verified exact DOP853 prefix:", reference.paths.directory)
print("Samples:", reference.times.size, "State shape:", reference.states.shape)
print("Maximum periodic discrepancy per particle:", distances.max(axis=1))
print("Maximum periodic discrepancy:", distances.max())


Verified exact DOP853 prefix: /home/juan/Proyectos/GC2D_intranet/data/trajectory/h5_three_radial_dop853_t35
Samples: 3501 State shape: (6, 3501)
Maximum periodic discrepancy per particle: [5.50447209e-06 2.82184553e-07 5.82193843e-08]
Maximum periodic discrepancy: 5.504472094580963e-06
